In [1]:
import StableDiffusion.ModelConverter
from StableDiffusion.DiffusionProcess import DiffusionProcess
device = 'cuda'
idleDevice = 'cpu'
filePath='../models/sd15models/v1-5-pruned-emaonly.ckpt'
diffusionDict = StableDiffusion.ModelConverter.load_from_standard_weights(input_file=filePath,\
                                                            device = 'cuda')
clipWeights=diffusionDict['clip']
diffusionWeights = diffusionDict['diffusion']
vaeEncoderWeights = diffusionDict['encoder']
vaeDecoderWeights = diffusionDict['decoder']

import torch
import StableDiffusion.ControlnetModelConverter
import importlib
importlib.reload(StableDiffusion.ControlnetModelConverter)
from StableDiffusion.ControlnetModelConverter import ControlnetModelConverter
filePath ="../models/ControlNet-v1-1/control_v11p_sd15_canny.pth" 
controlnetCannyDict  = torch.load(filePath, map_location= device)

import torch 
import StableDiffusion.VaeEncoder 
import StableDiffusion.VaeDecoder
import StableDiffusion.ClipEncoder
import StableDiffusion.DiffusionProcess
import importlib
importlib.reload(StableDiffusion.VaeEncoder)
importlib.reload(StableDiffusion.VaeDecoder)
importlib.reload(StableDiffusion.ClipEncoder)
importlib.reload(StableDiffusion.DiffusionProcess)
from StableDiffusion.VaeDecoder import VaeDecoder
from StableDiffusion.VaeEncoder import VaeEncoder
from StableDiffusion.ClipEncoder import ClipEncoder
from StableDiffusion.DiffusionProcess import DiffusionProcess
clipEncoder = ClipEncoder().to(device)
vaeEncoder = VaeEncoder().to(device)
vaeDecoder = VaeDecoder().to(device)
diffusionProcess = DiffusionProcess().to(device)
clipEncoder.load_state_dict(clipWeights,strict=True)
vaeEncoder.load_state_dict(vaeEncoderWeights ,strict=True)
vaeDecoder.load_state_dict(vaeDecoderWeights,strict=True)
diffusionProcess.load_state_dict(diffusionWeights,strict=True)



import StableDiffusion.ControlnetSD
import StableDiffusion.DiffusionProcessControlnet
importlib.reload(StableDiffusion.ControlnetSD)
importlib.reload(StableDiffusion.DiffusionProcessControlnet)
from StableDiffusion.ControlnetSD import ControlnetSD
from StableDiffusion.DiffusionProcessControlnet import DiffusionProcessControlnet


diffusionProcessControlnet = DiffusionProcessControlnet().to(device)
diffusionProcessControlnet.load_state_dict(diffusionWeights,strict=False)
diffusionProcessControlnet.loadControlnetWeightsDict(controlnetCannyDict)

/home/aistudio/external-libraries/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
import os
import wandb
import torch
import numpy as np
import matplotlib.pyplot as plt 
from IPython import display
import StableDiffusion.LoraDataSet
import StableDiffusion.DdpmSamplerTorch
import StableDiffusion.Utils
import StableDiffusion.LoraUtils
import StableDiffusion.ControlnetDataSet
import StableDiffusion.ControlnetUtils
import importlib
importlib.reload(StableDiffusion.DdpmSamplerTorch)
importlib.reload(StableDiffusion.LoraDataSet)
importlib.reload(StableDiffusion.Utils)
importlib.reload(StableDiffusion.LoraUtils)
importlib.reload(StableDiffusion.ControlnetDataSet)
importlib.reload(StableDiffusion.ControlnetUtils)
from StableDiffusion.Utils import Utils
from StableDiffusion.DdpmSamplerTorch import DdpmSamplerTorch
from StableDiffusion.LoraDataSet import LoraDataSet
from tqdm import tqdm
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from StableDiffusion.LoraUtils import *
from StableDiffusion.GenPipe import GenPipe
from StableDiffusion.ControlnetDataSet import ControlnetDataSet
from StableDiffusion.ControlnetUtils import ControlnetUtils

# wandb.init(
#     project='xiaofeixiang lora training ',
#     config={
#         'image nums':25,
#         'epoch num':5,
#         'learning rate':1e-4,
#         'batch size':5,
        
#     }
# )


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sampler = DdpmSamplerTorch()
sampler.to(device)
controlnetDataSet = ControlnetDataSet()
controlnetDataLoader = DataLoader(controlnetDataSet,batch_size=5,
                                shuffle=True,num_workers=0,drop_last=True,collate_fn=None)
vaeEncoder.eval()
vaeDecoder.eval()
clipEncoder.eval()
diffusionProcessControlnet.train()

trainableParamsDict = {}
for name,param in diffusionProcessControlnet.named_parameters():
    if param.requires_grad:
        #print(name,param.shape)
        trainableParamsDict[name]=param


# for name,param in trainableParamsDict.items():
#     print(name,param.shape)
    
lossHistory = []
epochLossHistory = []



optimizer = torch.optim.Adam(trainableParamsDict.values(), lr=1e-4)
EpochNum = 1
for epoch in tqdm(range(EpochNum),desc='Epoch'):
    epochLoss = 0.0
    for i,data in enumerate(tqdm(controlnetDataLoader)):
        img,controlHint,promptTokens,attentionMask = data
        #print(img.shape,controlHint.shape,promptTokens.shape,attentionMask.shape)
        #Utils.showBatchImage(controlHint)
        


ControlnetUtils.writeControlnetToFile(diffusionProcessControlnet)



Epoch: 100%|██████████| 1/1 [00:01<00:00,  1.47s/it]


controlnetOutput.input_blocks.0.0.weight torch.Size([320, 4, 3, 3])
controlnetOutput.input_blocks.0.0.bias torch.Size([320])
controlnetOutput.input_blocks.1.0.groupnorm_feature.weight torch.Size([320])
controlnetOutput.input_blocks.1.0.groupnorm_feature.bias torch.Size([320])
controlnetOutput.input_blocks.1.0.conv_feature.weight torch.Size([320, 320, 3, 3])
controlnetOutput.input_blocks.1.0.conv_feature.bias torch.Size([320])
controlnetOutput.input_blocks.1.0.linear_time.weight torch.Size([320, 1280])
controlnetOutput.input_blocks.1.0.linear_time.bias torch.Size([320])
controlnetOutput.input_blocks.1.0.groupnorm_merged.weight torch.Size([320])
controlnetOutput.input_blocks.1.0.groupnorm_merged.bias torch.Size([320])
controlnetOutput.input_blocks.1.0.conv_merged.weight torch.Size([320, 320, 3, 3])
controlnetOutput.input_blocks.1.0.conv_merged.bias torch.Size([320])
controlnetOutput.input_blocks.1.1.groupnorm.weight torch.Size([320])
controlnetOutput.input_blocks.1.1.groupnorm.bias torch

In [ ]:
import os
import wandb
import torch
import numpy as np
import matplotlib.pyplot as plt 
from IPython import display
import StableDiffusion.LoraDataSet
import StableDiffusion.DdpmSamplerTorch
import StableDiffusion.Utils
import StableDiffusion.LoraUtils
import importlib
importlib.reload(StableDiffusion.DdpmSamplerTorch)
importlib.reload(StableDiffusion.LoraDataSet)
importlib.reload(StableDiffusion.Utils)
importlib.reload(StableDiffusion.LoraUtils)
from StableDiffusion.Utils import Utils
from StableDiffusion.DdpmSamplerTorch import DdpmSamplerTorch
from StableDiffusion.LoraDataSet import LoraDataSet
from tqdm import tqdm
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from StableDiffusion.LoraUtils import *
from StableDiffusion.GenPipe import GenPipe

wandb.init(
    project='xiaofeixiang lora training ',
    config={
        'image nums':25,
        'epoch num':5,
        'learning rate':1e-4,
        'batch size':5,
        
    }
)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sampler = DdpmSamplerTorch()
sampler.to(device)
loraDataSet = LoraDataSet()
loraDataLoader = DataLoader(loraDataSet,batch_size=5,
                                shuffle=True,num_workers=0,drop_last=True,collate_fn=None)
# batchsize = 10 is almost the biggest batchsize
loraDataSet.__len__()
img,promtTokens,attentionMask = loraDataSet[0]
print(img.shape,promtTokens.shape,attentionMask.shape)

vaeEncoder.eval()
vaeDecoder.eval()
clipEncoder.eval()
diffusionProcess.train()

trainableParamsList = []
for name,param in diffusionProcess.named_parameters():
    if param.requires_grad:
        #print(name,param.shape)
        trainableParamsList.append(param)  


lossHistory = []
epochLossHistory = []



optimizer = torch.optim.Adam(trainableParamsList, lr=1e-4)
EpochNum = 100
for epoch in tqdm(range(EpochNum),desc='Epoch'):
    epochLoss = 0.0
    for i,data in enumerate(tqdm(loraDataLoader)):
        img,promptTokens,attentionMask = data
        with torch.no_grad():
            latentImg = vaeEncoder(img)
        
        with torch.no_grad():
            clipOutputsPositive = clipEncoder(promptTokens,attentionMask)
        
        
        BatchSize,Channel,Height,Width = latentImg.shape

        noiseLatent = torch.randn(BatchSize,Channel,Height,Width,dtype=torch.float32,device=device)
        timeSteps =torch.randint(0,1000,(BatchSize,),device=device)
        #print(timeSteps.shape,timeSteps)
        latentImgNoised = sampler.addNoiseBatchTrain(latentImg,noiseLatent,timeSteps)
        timeEmb320 =Utils.getTimeEmbeddingBatch(timeSteps)
        timeEmb320 = sampler.numpy2Tensor(timeEmb320)
        #print(latentImgNoised.shape,clipOutputsPositive.shape,timeEmb320.shape)
        predictedNoise= diffusionProcess(latentImgNoised,clipOutputsPositive,timeEmb320)
        with torch.no_grad():        
            noisedImg =vaeDecoder(latentImgNoised)
            
        loss = nn.functional.mse_loss(predictedNoise,noiseLatent)
        
        lossHistory.append(loss.item())
        epochLoss = epochLoss + loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f'epoch: {epoch}, iter: {i}, loss:{loss.item()}')
        
            
        #Utils.showBatchImage(img[:1])
        #Utils.showBatchImage(latentImg[:1,:3])
        #Utils.showBatchImage(noisedImg[:1])
    epochLoss = epochLoss / len(loraDataLoader)
    epochLossHistory.append(epochLoss)
    
    genPipe = GenPipe(vaeEncoder,vaeDecoder,clipEncoder,diffusionProcess)
    prompt = 'xiaofeixiang,1girl'
    prompt ='xiaofeixiang,1girl, solo, flower, long hair, \
    dress, bare shoulders, upper body, \
        realistic, lily \(flower\), black eyes, black hair, brown hair, holding flower, lips, sleeveless, \
    looking at viewer, holding, wind, closed mouth, bouquet, blue dress'
    promptNegative ='low quality, worst quality, blurry, out of focus, \
        jpeg artifacts, watermark,\
            text, logo, signature, oversharpened, \
                overexposed, underexposed, bad anatomy, \
                    bad proportions, deformed, distorted, \
                        disfigured, mutated, extra limbs, extra arms,\
                            extra legs, extra fingers, fused fingers,\
                                missing fingers, long neck, cross-eyed, \
                                    bad eyes, plastic skin, doll-like, cgi, 3d render'

        
          
    print(f'epoch: {epoch}, epochLoss:{epochLoss}')
    wandb.log({'epoch/loss': epochLoss,
               'epoch/iter': epoch,
               #'epoch/test lora image':wandb.Image(testLoraImg,caption=f'test lora {epoch}')
               })
    
    if epoch % 5 ==0:
        GenNum=1
        for i in range(GenNum):
            genPipe.seed = np.random.randint(10000)
            genPipe.numInferenceSteps =20
            imgStepList = genPipe.genImage(prompt,promptNegative)
            imgBatch = imgStepList[-1]
            testLoraImg = Utils.getBatchImage(imgBatch)
        
        wandb.log({'epoch/test lora image':wandb.Image(testLoraImg,caption=f'test lora {epoch}')})    
    display.clear_output(wait=True)
    plt.figure(figsize= (10,5))
    plt.plot(epochLossHistory,label='epoch loss')    
    display.display(plt.gcf())
    plt.close()
           
writeLoraToFile(diffusionProcess)

wandb.finish()
        